# Fine-tune a Udgam Labs model with LoRA

One recipe for every Udgam Labs model in the transformers format: [Udgam-384M-Chat](https://huggingface.co/udgamlabs/Udgam-384M-Chat),
[Laghu](https://huggingface.co/udgamlabs/Laghu), [Laghu-Base](https://huggingface.co/udgamlabs/Laghu-Base),
[Gargi](https://huggingface.co/udgamlabs/Gargi), [Vemana-Base](https://huggingface.co/udgamlabs/Vemana-Base) and
[Lilavati](https://huggingface.co/udgamlabs/Lilavati). They are small (150M-400M parameters), so this runs on a free
Colab GPU, an Apple Silicon Mac or a laptop CPU in minutes.

LoRA trains a few small matrices next to the frozen model (about 1-3% extra parameters). The result is an adapter of a
few MB that loads on top of the base model, or merges into it.

The example task turns a restaurant booking request into JSON. It is built in the notebook, so it runs as is; replace it
with your own rows (`prompt` and `completion`). The last cells score held-out requests the model never saw, save the
adapter, and show how to publish it with `base_model` set, so your model links back to the one it came from.

In [ ]:
%pip install -q "transformers>=5.0" "peft>=0.17" torch accelerate

In [ ]:
MODEL = "udgamlabs/Laghu"   # or Udgam-384M-Chat, Laghu-Base, Gargi, Vemana-Base, Lilavati
STEPS, BATCH, ACCUM, LR, RANK = 150, 8, 1, 2e-4, 16   # on a Mac or a small GPU: BATCH, ACCUM = 4, 2
OUT = "my-lora-adapter"

## The data
Each row is a `prompt` and the `completion` the model should learn. Chat models get the row through their chat template;
base models (Vemana-Base, Laghu-Base, Lilavati) get plain text. The loss is computed on the completion only.

In [ ]:
import json, random

rnd = random.Random(0)
NAMES = ["Asha", "Ravi", "Meera", "John", "Fatima", "Arjun", "Li", "Sara", "Kabir", "Nina", "Omar", "Priya"]
DAYS = ["today", "tomorrow", "Friday", "Saturday", "Sunday", "Monday"]
TIMES = [("7pm", "19:00"), ("8 pm", "20:00"), ("half past seven", "19:30"), ("1pm", "13:00"), ("noon", "12:00"), ("9:15 pm", "21:15")]
WORDS = {2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 8: "eight"}
FORMS = ["Book a table for {n} {day} at {t}, name {name}.",
         "Hi, can I get a table for {n} people at {t} {day}? It's {name}.",
         "{name} here - {n} of us, {day}, around {t} please.",
         "Reserve for {name}: party of {n}, {t} on {day}."]

def row():
    k, (t, hhmm), day, name = rnd.choice(list(WORDS)), rnd.choice(TIMES), rnd.choice(DAYS), rnd.choice(NAMES)
    n = WORDS[k] if rnd.random() < 0.5 else str(k)
    prompt = rnd.choice(FORMS).format(n=n, day=day, t=t, name=name)
    return {"prompt": "Extract the booking as JSON.\n" + prompt,
            "completion": json.dumps({"name": name, "people": k, "day": day, "time": hhmm})}

rows = [row() for _ in range(240)]
train, test = rows[:200], rows[200:]
print(train[0])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(device)
# chat models use their chat template; base models (Laghu-Base, Vemana-Base, Lilavati) use plain text: a base model's
# turn markers were never trained, so it would not learn to stop
chat = tok.chat_template is not None and "base" not in MODEL.lower()
print(MODEL, f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters,", "chat template" if chat else "plain text", "on", device)

def render(r, with_answer=True):
    """(prompt ids, completion ids): the chat template for chat models, plain text for base models"""
    msgs = [{"role": "user", "content": r["prompt"]}]
    if chat:   # the completion is the template's own assistant turn, end-of-turn marker included
        p = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        full = tok.apply_chat_template(msgs + [{"role": "assistant", "content": r["completion"]}], tokenize=False)
        assert full.startswith(p), "chat template: the prompt is not a prefix of the full conversation"
        c = full[len(p):]
    else:
        p, c = r["prompt"] + "\nJSON: ", r["completion"] + "\n" + (tok.eos_token or "")
    p_ids = tok(p, add_special_tokens=not chat)["input_ids"]
    c_ids = tok(c, add_special_tokens=False)["input_ids"] if with_answer else []
    return p_ids, c_ids

In [ ]:
from peft import LoraConfig, get_peft_model

torch.manual_seed(0)   # LoRA's starting weights; with the data seed above, a rerun on the same device repeats
lora = LoraConfig(r=RANK, lora_alpha=2 * RANK, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
def batch(rs):
    seqs = [render(r) for r in rs]
    L = max(len(p) + len(c) for p, c in seqs)
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    ids = torch.full((len(rs), L), pad); lab = torch.full((len(rs), L), -100); att = torch.zeros((len(rs), L), dtype=torch.long)
    for i, (p, c) in enumerate(seqs):
        s = p + c
        ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
        lab[i, len(p):len(s)] = torch.tensor(c)   # loss on the completion only
    return ids.to(device), att.to(device), lab.to(device)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / 10) * max(0.0, 1 - s / STEPS))
model.train()
for step in range(STEPS):
    for _ in range(ACCUM):
        ids, att, lab = batch(rnd.sample(train, BATCH))
        loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
        (loss / ACCUM).backward()
    opt.step(); sched.step(); opt.zero_grad()
    if step % 25 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  loss {loss.item():.3f}")

## Check it on requests it never saw
Greedy decoding on the 40 held-out rows; a row counts only when the JSON parses and every field matches.

In [ ]:
model.eval()
# stop on end-of-text and on the special tokens that close every training answer (a chat template's turn end)
stop = sorted({tok.eos_token_id, *[t for t in render(train[0])[1] if t in tok.all_special_ids]} - {None})

def answer(r, max_new_tokens=48):
    p, _ = render(r, with_answer=False)
    x = torch.tensor([p], device=device)
    with torch.no_grad():
        y = model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=stop,
                           pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
    return tok.decode(y[0, len(p):], skip_special_tokens=True).strip()

def exact(r):
    try:   # the first JSON object in the answer
        return json.JSONDecoder().raw_decode(answer(r))[0] == json.loads(r["completion"])
    except ValueError:
        return False

right = sum(exact(r) for r in test)
print(f"held-out exact match: {right}/{len(test)}")
print(test[0]["prompt"].split("\n")[1], "->", answer(test[0]))

## Save, reload, publish
`save_pretrained` writes the adapter (`adapter_config.json` names the base model). To publish it, `push_to_hub` writes a
model card with `base_model: <the Udgam Labs model>` and `library_name: peft`, so Hugging Face lists your adapter under
the model it came from. `merge_and_unload()` gives one plain model instead, for GGUF or MLX conversion.

In [ ]:
model.save_pretrained(OUT); tok.save_pretrained(OUT)
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(device)
again = PeftModel.from_pretrained(base, OUT).eval()
print(json.load(open(f"{OUT}/adapter_config.json"))["base_model_name_or_path"], "- adapter reloaded")

# model.push_to_hub("your-name/my-booking-adapter")     # base_model is set from MODEL
# merged = again.merge_and_unload(); merged.save_pretrained("merged")